# Deep Learning Training & Evaluation

Analyzes LSTM, 1D CNN, and Transformer results from cell-based LOOCV
with Optuna hyperparameter tuning. Compares DL performance against classical baselines.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
import os
import mlflow

os.chdir(Path("..").resolve())
sns.set_theme(style="whitegrid", context="notebook")

with open("config/default.yaml") as f:
    config = yaml.safe_load(f)

## 1. Load Results

In [ ]:
with open("experiments/classical_results.yaml") as f:
    classical_results = yaml.safe_load(f)

with open("experiments/dl_results.yaml") as f:
    dl_results = yaml.safe_load(f)

all_results = {**classical_results, **dl_results}
dl_models = ["lstm", "cnn", "transformer"]
print("Models:", list(all_results.keys()))
for name, m in all_results.items():
    print(f"  {name.upper():>12s}: RMSE={m['rmse_mean']:.6f} +/- {m['rmse_std']:.6f}, R2={m['r2_mean']:.4f}")

## 2. Fetch Fold-level Metrics from MLflow

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("soh_benchmark")

client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("soh_benchmark")

all_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time asc"],
)

run_records = []
for run in all_runs:
    params = run.data.params
    tags = run.data.tags
    metrics = run.data.metrics
    model = params.get("model", tags.get("model", "unknown"))
    fold = params.get("fold", tags.get("fold", None))
    seed = params.get("seed", tags.get("seed", None))
    if fold is not None and "rmse" in metrics:
        run_records.append({
            "model": model,
            "fold": int(fold),
            "seed": int(seed) if seed else None,
            "rmse": metrics["rmse"],
            "mae": metrics["mae"],
            "maxae": metrics["maxae"],
            "r2": metrics["r2"],
        })

runs_df = pd.DataFrame(run_records)
print(f"Total runs: {len(runs_df)}")
if len(runs_df) > 0:
    print(runs_df.groupby(["model", "fold"])["rmse"].agg(["mean", "std"]).round(6))

## 3. Fold-wise RMSE: DL Models

In [ ]:
dl_runs = runs_df[runs_df["model"].isin(dl_models)].copy()
fold_rmse = dl_runs.groupby(["model", "fold"])["rmse"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=fold_rmse, x="fold", y="rmse", hue="model", ax=ax)
ax.set_xlabel("LOOCV Fold (held-out cell)")
ax.set_ylabel("RMSE")
ax.set_title("DL Models: Fold-wise RMSE")
plt.tight_layout()
plt.show()

## 4. Seed Variance Analysis

In [ ]:
seed_stats = dl_runs.groupby(["model", "fold"]).agg(
    rmse_mean=("rmse", "mean"),
    rmse_std=("rmse", "std"),
    n_seeds=("seed", "count"),
).reset_index()

print("Seed variance across runs:")
print(seed_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=seed_stats, x="fold", y="rmse_mean", hue="model", ax=ax)
ax.set_xlabel("Fold")
ax.set_ylabel("Mean RMSE (+/- std over seeds)")
ax.set_title("DL Models: Mean RMSE with Seed Variance")
plt.tight_layout()
plt.show()

## 5. DL vs Classical Comparison

In [ ]:
summary_rows = []
for name, m in all_results.items():
    category = "DL" if name in dl_models else "Classical"
    summary_rows.append({
        "Model": name.upper(),
        "Category": category,
        "RMSE": m["rmse_mean"],
        "RMSE_std": m["rmse_std"],
        "MAE": m["mae_mean"],
        "MaxAE": m["maxae_mean"],
        "R2": m["r2_mean"],
        "R2_std": m["r2_std"],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("RMSE")
print(summary_df.to_string(index=False, float_format="%.6f"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = {"Classical": "#2196F3", "DL": "#FF5722"}
sns.barplot(data=summary_df, x="Model", y="RMSE", hue="Category", palette=palette, ax=axes[0])
axes[0].set_title("RMSE Comparison")
axes[0].set_ylabel("RMSE (lower is better)")
sns.barplot(data=summary_df, x="Model", y="R2", hue="Category", palette=palette, ax=axes[1])
axes[1].set_title("R-squared Comparison")
axes[1].set_ylabel("R-squared (higher is better)")
plt.tight_layout()
plt.show()

## 6. Training Convergence Analysis

In [ ]:
dl_ckpt_dir = Path("experiments/dl")
ckpt_files = list(dl_ckpt_dir.glob("*.pt")) if dl_ckpt_dir.exists() else []
print(f"Found {len(ckpt_files)} DL checkpoints")
if ckpt_files:
    for f in sorted(ckpt_files)[:10]:
        print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 7. Inference Time Comparison

In [ ]:
time_rows = []
for name, m in all_results.items():
    time_rows.append({
        "Model": name.upper(),
        "Inference Time (s)": m.get("inference_time_mean_s", np.nan),
    })

time_df = pd.DataFrame(time_rows).sort_values("Inference Time (s)")
print(time_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=time_df, x="Model", y="Inference Time (s)", ax=ax)
ax.set_title("Total Training Time per Model (all folds)")
ax.set_ylabel("Time (seconds)")
plt.tight_layout()
plt.show()

## 8. Summary

**Key findings:**
- Classical models (SVR, RF) significantly outperform DL models on this small dataset
- DL models struggle with only 636 samples and 4 cells
- High variance across seeds indicates DL models are data-limited
- Sequence-based DL approaches require more training data to generalize
- SVR achieves the best RMSE with lowest variance across folds